In [7]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append("..")
from data.base import get_connection

# 1. Load the working geocoded dataset (4,529 unique intersections)
yearly_cams = pd.read_csv('../data/yearly_camera_coordinates.csv')

# Register it in DuckDB so we can join directly on the exact coordinates and search_query
con = get_connection()
try:
    con.register('yearly_cams_ref', yearly_cams)
    
    # Calculate operational metrics per camera location and type
    # A camera active across multiple calendar years or consistently within a year is permanent.
    query = """
        SELECT 
            search_query,
            camera_type,
            violation_code,
            FIRST(Latitude) AS Latitude,
            FIRST(Longitude) AS Longitude,
            MIN(year) AS first_active_year,
            MAX(year) AS last_active_year,
            COUNT(DISTINCT year) AS active_years_count,
            -- Check if active in recent operational year (2026)
            MAX(CASE WHEN year = 2026 THEN 1 ELSE 0 END) AS active_in_2026
        FROM yearly_cams_ref
        GROUP BY search_query, camera_type, violation_code
    """
    camera_catalog = con.sql(query).df()
finally:
    con.close()

print(f"Total deduplicated camera enforcement points: {len(camera_catalog):,}")

# 2. Deployment Permanence Logic:
# - Red Light cameras: always fixed physical infrastructure -> Permanent
# - Speed cameras:
#     * Active across 2 or more distinct years -> Permanent fixed location
#     * Active in only 1 year -> Candidate for mobile rotation or newly installed unit
def classify_deployment(row):
    if row['camera_type'] == 'Red Light':
        return 'Permanent'
    
    # Speed camera evaluation:
    if row['active_years_count'] >= 2:
        return 'Permanent'
    elif row['first_active_year'] == 2026:
        # Newly deployed permanent or mobile unit in 2026
        return 'Newly Installed (2026)'
    else:
        # Only observed in a single prior year (e.g., visited in 2018 and never returned)
        return 'Mobile / Temporary'

camera_catalog['deployment_category'] = camera_catalog.apply(classify_deployment, axis=1)

print("\n=== Deployment Classification Summary ===")
print(pd.crosstab(camera_catalog['deployment_category'], camera_catalog['camera_type'], margins=True))

# 3. Filter current active cameras (active in 2026, excluding single-year historical mobile units)
current_cameras_2026 = camera_catalog[
    (camera_catalog['active_in_2026'] == 1) & 
    (camera_catalog['deployment_category'] != 'Mobile / Temporary')
]

print("\n=== Current Active Cameras (2026 Network Footprint) ===")
print(current_cameras_2026['camera_type'].value_counts())

Total deduplicated camera enforcement points: 4,610

=== Deployment Classification Summary ===
camera_type             Red Light  Speed   All
deployment_category                           
Mobile / Temporary              0    428   428
Newly Installed (2026)          0    116   116
Permanent                     617   3449  4066
All                           617   3993  4610

=== Current Active Cameras (2026 Network Footprint) ===
camera_type
Speed        2045
Red Light     591
Name: count, dtype: int64


In [ ]:
import pandas as pd

# Option A: Strictly multi-year permanent cameras active in 2026
current_permanent_strict = camera_catalog[
    (camera_catalog['active_in_2026'] == 1) & 
    (camera_catalog['deployment_category'] == 'Permanent')
]

# Option B: All 2026 active cameras excluding historical mobile/temporary units
# (includes multi-year permanent + newly installed 2026 sites)
current_active_all = camera_catalog[
    (camera_catalog['active_in_2026'] == 1) & 
    (camera_catalog['deployment_category'] != 'Mobile / Temporary')
]

# Save Option B as your primary June 2026 operational inventory
output_path = '../data/current_active_permanent_cameras_2026.csv'
current_active_all[[
    'search_query', 
    'camera_type', 
    'violation_code', 
    'Latitude', 
    'Longitude', 
    'deployment_category',
    'first_active_year', 
    'last_active_year', 
    'active_years_count'
]].to_csv(output_path, index=False)

print(f"Exported current active cameras to: {output_path}")
print(f"Total active sites: {len(current_active_all):,}")
print(current_active_all['camera_type'].value_counts())

Exported current active cameras to: ../data/current_active_permanent_cameras_2026.csv
Total active sites: 2,636
camera_type
Speed        2045
Red Light     591
Name: count, dtype: int64


: 